# 06 · Fusion training — EHR encoder → FiLM → head, Distribution-Balanced Loss

The research question (§0): **do EHR features complement CXR embeddings, especially on
rare labels where the image signal is weak?** This notebook answers it by training a
head on the frozen `x_cxr` from 05, optionally conditioned on EHR via FiLM, and
comparing `none → concat → film` on per-label AUPRC with a confound-corrected Δ.

**Everything is frozen upstream.** `x_cxr` (05) and the EHR features (02) are cached;
this notebook trains only the small fusion stack, so a full config is seconds-to-minutes,
not hours. That is what makes the D1–D5 ablation grid affordable.

**Architecture (plan §B5–B7):**

```
EHR (02)  ── EmbeddingBag(org/spec, sum) + emb(setting,recency,status) + labs⊕mask
             → MLP → e[128]
                        │
x_cxr[768] ── FiLMGenerator(e) → (γ,β) → norm(γ⊙x+β) = h ── head → 13 logits
```

**Two design points where the plan is silent, resolved here:**

1. **`fusion="none"` applies the same LayerNorm as FiLM** (`head(norm(x))`, not
   `head(x)`). The plan's E2.1 gate demands *step-0 FiLM logits == CXR-only logits*.
   That is only literally true if both share the pre-head transform; with identity
   init (γ=1,β=0) FiLM reduces to `norm(x)`, so CXR-only must be `norm(x)` too. This
   makes the anti-fusion-gap guarantee exact rather than approximate.
2. **DB Loss is the same `ResampleLoss` port used in 04b** (§B8), math verbatim from
   `wutong16/DistributionBalancedLoss`, `class_freq.pkl` from 02.

**Prereq:** 05 done (`study_x_cxr.npy`, `x_cxr_train_stats.npz`) and 02 done
(`cohort_study_level.parquet`, `ehr_vocab.pkl`, `class_freq.pkl`). CPU-trainable, but
a GPU makes the ablation sweep snappy.


## C0 · Config — one `Cfg` drives every ablation

Every D1–D5 knob is a field here. `run_config(cfg)` reads only this object, so a sweep
is a list of `Cfg`s. Hyperparameters follow §B/§C; where the plan is silent, common
defaults (GELU, LayerNorm, dropout 0.2/0.1, AdamW cosine+warmup) — Tuan tunes later.

In [1]:
import os, io, time, json, math, pickle, random
from dataclasses import dataclass, field, replace
from typing import Tuple
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
CACHE_DIR = os.path.join(DATA_DIR, 'cxr_cls_cache')
CKPT_DIR  = os.path.join(DATA_DIR, 'fusion_v2'); os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')

@dataclass
class Cfg:
    # ---- D1 fusion architecture ----
    fusion: str          = 'film'          # {'none','concat','film'}
    film_gamma_mode: str = 'full'          # {'full','gating_only'}   full = 1+tanh, identity-able
    film_identity_init: bool = True        # zero last gen layer -> gamma=1,beta=0 at t0
    head_input: str      = 'h'             # {'h','h_concat_e'}   (D1-A5)
    film_reg: float      = 0.0             # optional ||gamma-1||,||beta|| penalty (B6)
    # ---- D2 EHR encoding ----
    ehr_cat_encoding: str = 'embedding'    # {'embedding','onehot_mlp'}
    # ---- D3 micro tiers ----
    micro_tiers: Tuple[int,...] = (1,2,3)  # subset of (1,2,3,4); () = labs only
    # ---- new EHR blocks (02 C7-bis / C7-ter) -- switches, not welds ----
    use_vitals: bool = True                # temp/HR/RR/SpO2/BP; acute physiology like labs
    use_como: bool   = True                # diabetes/HIV. Carries the A4 report-writing
                                           # confound -> must stay independently ablatable.
    # ---- D5 loss ----
    loss: str            = 'db'            # {'db','bce_posweight','asl'}  ('ral' TODO)
    # ---- dims (B5/B7) ----
    d_cxr: int = 768
    d_e: int   = 128                       # keep small (B5): wide e memorizes patient noise
    d_hidden_enc: int = 256
    d_hidden_head: int = 256
    dropout_enc: int = 0.2
    dropout_head: int = 0.1
    emb_org: int = 16; emb_spec: int = 8; emb_small: int = 4   # per-field EmbeddingBag dims
    # ---- optimization (Part C) ----
    epochs: int   = 25                     # hard cap; the head peaks ~ep5 and then overfits
    patience: int = 6                      # early-stop after N epochs with no val-mAP gain (0=off)
    batch: int  = 512
    lr: float   = 1e-3
    wd: float   = 1e-2                     # weights; norms/biases get 0 (param groups)
    warmup_frac: float = 0.05
    grad_clip: float = 5.0
    seed: int   = 42                       # per-run: C12 varies this to get error bars
    # ---- bookkeeping ----
    tag: str = 'film_default'

BASE = Cfg()
print('base cfg:', BASE)

Mounted at /content/drive
device: cuda | NVIDIA RTX PRO 6000 Blackwell Server Edition
base cfg: Cfg(fusion='film', film_gamma_mode='full', film_identity_init=True, head_input='h', film_reg=0.0, ehr_cat_encoding='embedding', micro_tiers=(1, 2, 3), use_vitals=True, use_como=True, loss='db', d_cxr=768, d_e=128, d_hidden_enc=256, d_hidden_head=256, dropout_enc=0.2, dropout_head=0.1, emb_org=16, emb_spec=8, emb_small=4, epochs=25, patience=6, batch=512, lr=0.001, wd=0.01, warmup_frac=0.05, grad_clip=5.0, seed=42, tag='film_default')


## C1 · Load the frozen inputs — `x_cxr`, EHR table, vocab, class_freq

`x_cxr` is joined to the cohort by `study_id` and **standardized with the train-only
stats 05 saved** (§B1: stats from train, applied to all). The EHR parquet and vocab
carry the §A8 schema exactly as 02 emitted it; `class_freq.pkl` is the sole input DB
Loss takes, and its label order is asserted against the cohort's.

In [2]:
# --- cohort labels/order (the master study order for this notebook) ---
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS) == 13

ehr = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'))
ehr = ehr.sort_values('study_id').reset_index(drop=True)
N   = len(ehr)
print(f'EHR table: {N:,} studies x {ehr.shape[1]} cols')

# --- x_cxr, aligned to ehr.study_id ---
X    = np.load(os.path.join(CACHE_DIR, 'study_x_cxr.npy'))
xidx = pd.read_parquet(os.path.join(CACHE_DIR, 'study_x_cxr_index.parquet'))
row_of = dict(zip(xidx.study_id.to_numpy(), xidx.row.to_numpy()))
assert set(ehr.study_id) <= set(row_of), 'some cohort study has no x_cxr -> re-run 05'
order = ehr.study_id.map(row_of).to_numpy()
X = X[order]                                            # now row i of X == study ehr.study_id[i]

st = np.load(os.path.join(CACHE_DIR, 'x_cxr_train_stats.npz'))
X = ((X - st['mean']) / st['std']).astype('float32')    # train-only standardization (B1)
print(f'x_cxr aligned+standardized: {X.shape}  mean~{X.mean():.3f} std~{X.std():.3f}')

# --- vocab + class_freq ---
vocab = pickle.load(open(os.path.join(OUT_DIR, 'ehr_vocab.pkl'), 'rb'))
ORG_V, SPEC_V = vocab['org_vocab_size'], vocab['spec_vocab_size']
RES = vocab['reserved']; MISSING = RES['MISSING']
print(f"org vocab {ORG_V} | spec vocab {SPEC_V} | reserved {RES}")

cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
assert cf.get('labels', LABELS) == LABELS, 'class_freq label order != cohort -> DB Loss mis-weights'
CLASS_FREQ, NUM_TRAIN = np.asarray(cf['class_freq']), cf['num_train']
TRAIN_PREV = CLASS_FREQ / NUM_TRAIN
TIER = np.where(TRAIN_PREV < 0.02, 'tail', np.where(TRAIN_PREV < 0.10, 'medium', 'head'))
print('label tiers:', dict(zip(LABELS, TIER)))

split = ehr.split.to_numpy()
Y = ehr[LABELS].to_numpy('float32')
tr_mask = split == 'train'
print('splits:', {s:int((split==s).sum()) for s in ['train','val','test']})

EHR table: 211,130 studies x 88 cols
x_cxr aligned+standardized: (211130, 768)  mean~-0.000 std~1.000
org vocab 311 | spec vocab 82 | reserved {'PAD': 0, 'UNK': 1, 'MISSING': 2, 'NO_GROWTH': 3, 'CANCELLED': 4, 'NEGATIVE': 5}
label tiers: {'Atelectasis': np.str_('head'), 'Cardiomegaly': np.str_('head'), 'Consolidation': np.str_('medium'), 'Edema': np.str_('head'), 'Enlarged_Cardiomediastinum': np.str_('medium'), 'Fracture': np.str_('medium'), 'Lung_Lesion': np.str_('medium'), 'Lung_Opacity': np.str_('head'), 'No_Finding': np.str_('head'), 'Pleural_Effusion': np.str_('head'), 'Pleural_Other': np.str_('tail'), 'Pneumonia': np.str_('head'), 'Pneumothorax': np.str_('medium')}
splits: {'train': 147791, 'val': 31670, 'test': 31669}


## C2 · Featurize the §A8 schema → in-memory tensors

Everything is tiny (211k × a few hundred dims), so it lives in RAM. Ragged organism /
specimen id lists become `EmbeddingBag` inputs (kept as per-study lists here, flattened
in the collate). Small categoricals (`setting`, `t3_recency_bin`, `t1_status`) → integer
codes for `nn.Embedding`. Counts (`t2_n_pending`, `t4_*`) are `log1p`-then-z-scored on
**train only**. Lab `*_value`/`*_mask` are already standardized by 02 — used as-is.

In [3]:
def as_id_lists(col):
    """parquet list-column -> list of 1-D int64 arrays (empty -> [MISSING])."""
    out = []
    for v in ehr[col].to_numpy():
        a = np.asarray(v, dtype=np.int64).ravel() if v is not None and np.ndim(v) else np.array([], np.int64)
        out.append(a if a.size else np.array([MISSING], np.int64))
    return out

RAGGED = {'t1_org':('t1_org_ids',ORG_V), 't1_spec':('t1_spec_ids',SPEC_V),
          't2_spec':('t2_spec_ids',SPEC_V), 't3_org':('t3_org_ids',ORG_V)}
bags = {k: as_id_lists(c) for k,(c,_) in RAGGED.items()}

def codes(series, cats):
    # case-insensitive: 02 stores setting as 'ed' (lowercase); a literal 'ED' match
    # silently dropped all 77,791 ED studies to code 0 (=inpatient), corrupting the
    # setting feature for 37% of the cohort.
    m = {c.lower(): i for i,c in enumerate(cats)}
    return series.astype(str).str.lower().map(lambda x: m.get(x, 0)).to_numpy('int64'), len(cats)

set_id,  N_SET    = codes(ehr['setting'],        ['inpatient','ED','outpatient'])
rec_id,  N_REC    = codes(ehr['t3_recency_bin'], ['none','7-30d','30-90d'])
stat_id, N_STAT   = ehr['t1_status'].to_numpy('int64'), 4     # 0..3 already int

BIN = ['t2_pending','t2_time_imprecise','t1_same_encounter']
binv = {b: ehr[b].to_numpy('float32') for b in BIN}

def zscore_train(x, use_log=True):
    x = x.astype('float32').copy()
    z = np.log1p(np.clip(x,0,None)) if use_log else x
    mu, sd = z[tr_mask].mean(), z[tr_mask].std() or 1.0
    return ((z - mu)/sd).astype('float32')

COUNTS = ['t2_n_pending','t3_n_specimens','t4_n_prior_positive_ever',
          't4_years_since_first_culture','t4_n_prior_encounters']
cnts = {c: zscore_train(ehr[c].to_numpy()) for c in COUNTS}

# labs: value+mask already standardized/derived by 02
LABS = vocab['labs']
lab_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in LABS], 1)
lab_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in LABS], 1)

# vitals: identical value+mask contract (02 C7-bis). vocab.get() so a pre-vitals pickle
# still loads -- an empty block simply contributes 0 dims.
VITALS = [c for c in vocab.get('vitals', []) if f'{c}_value' in ehr.columns]
_z = lambda k: np.zeros((N, k), 'float32')
vit_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in VITALS], 1) if VITALS else _z(0)
vit_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in VITALS], 1) if VITALS else _z(0)

# comorbidity: indicators + como_mask. como_mask=1 iff a prior CLOSED admission exists,
# i.e. iff a code could have been observed at all -- without it, "no diabetes" and
# "never admitted before" collapse to the same 0 (the NO_GROWTH failure, again).
COMO_COLS = [c for c in (vocab.get('comorbidity', []) + ['como_mask']) if c in ehr.columns]
como_arr  = np.stack([ehr[c].to_numpy('float32') for c in COMO_COLS], 1) if COMO_COLS else _z(0)
print(f'vitals: {len(VITALS)} -> {2*len(VITALS)} dims | comorbidity: {COMO_COLS}')

# recency value+mask (already standardized by 02), grouped by the tier that owns them
REC1 = ['t1_days_since_last_specimen','t1_days_since_last_positive']
REC3 = ['t3_days_since_last_positive']
def rec_block(names):
    v = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in names], 1)
    m = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in names], 1)
    return v, m
rec1_v, rec1_m = rec_block(REC1)
rec3_v, rec3_m = rec_block(REC3)

print('featurized. dims -> labs', lab_val.shape[1], '| counts', len(COUNTS),
      '| set/rec/status cats', (N_SET,N_REC,N_STAT))

vitals: 6 -> 12 dims | comorbidity: ['como_diabetes', 'como_hiv', 'como_mask']
featurized. dims -> labs 7 | counts 5 | set/rec/status cats (3, 3, 4)


## C3 · Dataset + collate — ragged ids → `EmbeddingBag` (flat + offsets)

`EmbeddingBag` takes a flat concatenation of every study's ids plus an `offsets`
vector marking where each study starts — the dense equivalent of multi-hot × weight,
without materializing the multi-hot (§B5). The collate builds those on the fly.

In [4]:
class FusionDS(Dataset):
    def __init__(self, idx): self.idx = np.asarray(idx)
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        item = dict(
            x=torch.from_numpy(X[i]), y=torch.from_numpy(Y[i]),
            set_id=set_id[i], rec_id=rec_id[i], stat_id=stat_id[i],
            lab_val=torch.from_numpy(lab_val[i]), lab_mask=torch.from_numpy(lab_mask[i]),
            rec1_v=torch.from_numpy(rec1_v[i]), rec1_m=torch.from_numpy(rec1_m[i]),
            rec3_v=torch.from_numpy(rec3_v[i]), rec3_m=torch.from_numpy(rec3_m[i]),
            vit_val=torch.from_numpy(vit_val[i]), vit_mask=torch.from_numpy(vit_mask[i]),
            como=torch.from_numpy(como_arr[i]),
            row=i)
        for b in BIN: item[b] = binv[b][i]
        for c in COUNTS: item[c] = cnts[c][i]
        for name in RAGGED: item[name] = torch.from_numpy(bags[name][i])
        return item

def collate(batch):
    out = dict(
        x=torch.stack([b['x'] for b in batch]),
        y=torch.stack([b['y'] for b in batch]),
        set_id=torch.tensor([b['set_id'] for b in batch]),
        rec_id=torch.tensor([b['rec_id'] for b in batch]),
        stat_id=torch.tensor([b['stat_id'] for b in batch]),
        lab_val=torch.stack([b['lab_val'] for b in batch]),
        lab_mask=torch.stack([b['lab_mask'] for b in batch]),
        rec1_v=torch.stack([b['rec1_v'] for b in batch]),
        rec1_m=torch.stack([b['rec1_m'] for b in batch]),
        rec3_v=torch.stack([b['rec3_v'] for b in batch]),
        rec3_m=torch.stack([b['rec3_m'] for b in batch]),
        vit_val=torch.stack([b['vit_val'] for b in batch]),
        vit_mask=torch.stack([b['vit_mask'] for b in batch]),
        como=torch.stack([b['como'] for b in batch]),
        row=torch.tensor([b['row'] for b in batch]))
    for b in BIN:    out[b] = torch.tensor([x[b] for x in batch], dtype=torch.float32)
    for c in COUNTS: out[c] = torch.tensor([x[c] for x in batch], dtype=torch.float32)
    for name in RAGGED:                                    # flat ids + offsets per bag
        lens = [len(x[name]) for x in batch]
        out[name] = (torch.cat([x[name] for x in batch]) if sum(lens)
                     else torch.empty(0, dtype=torch.int64),
                     torch.tensor([0]+lens[:-1]).cumsum(0))
    return out

def loader(idx, shuffle, bs):
    # num_workers=0: every feature is already an in-RAM numpy array, so worker processes
    # add only fork/IPC overhead (slower) -- and predict() rebuilds a loader each epoch,
    # whose worker teardown prints a screenful of harmless 'can only test a child process'
    # AssertionErrors from __del__. 0 workers is both faster here and silent.
    return DataLoader(FusionDS(idx), batch_size=bs, shuffle=shuffle, collate_fn=collate,
                      num_workers=0, pin_memory=(dev=='cuda'), drop_last=shuffle)

idx_tr = np.where(split=='train')[0]; idx_va = np.where(split=='val')[0]
idx_te = np.where(split=='test')[0]
print('loaders ready:', len(idx_tr), len(idx_va), len(idx_te))

loaders ready: 147791 31670 31669


## C4 · EHR encoder (§B5)

`EmbeddingBag(mode='sum')` for the ragged organism/specimen fields; `nn.Embedding` for
the small categoricals; continuous labs/recency/counts pass through as-is. The active
feature set is gated by `cfg.micro_tiers`, so `micro_tiers=()` is a labs-only encoder
(the D3 floor) and each added tier concatenates its block — the input dim is computed
from exactly the active pieces. LayerNorm throughout (§B: per-patient conditioning
forbids BatchNorm).

In [5]:
Act = nn.GELU

class EhrEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg; T = set(cfg.micro_tiers)
        self.T = T
        oh = (cfg.ehr_cat_encoding == 'onehot_mlp')
        def bag(vsize, dim):
            # onehot_mlp: multi-hot -> Linear (identical capacity, B5/D2). embedding: EmbeddingBag.
            return ('linear', nn.Linear(vsize, dim, bias=False)) if oh else \
                   ('bag', nn.EmbeddingBag(vsize, dim, mode='sum'))
        self.mods = nn.ModuleDict(); self.kinds = {}; d = 0
        def add(key, vsize, dim):
            kind, m = bag(vsize, dim); self.mods[key] = m; self.kinds[key] = kind; return dim
        # ---- always: labs value+mask, setting ----
        self.set_emb = nn.Embedding(3, cfg.emb_small); d += cfg.emb_small
        d += 2*len(vocab['labs'])                                    # lab value + mask
        self.n_vit  = len(VITALS)   if cfg.use_vitals else 0
        self.n_como = len(COMO_COLS) if cfg.use_como  else 0
        d += 2*self.n_vit + self.n_como                              # vitals value+mask, como
        # ---- tier 1 ----
        if 1 in T:
            d += add('t1_org',  ORG_V,  cfg.emb_org)
            d += add('t1_spec', SPEC_V, cfg.emb_spec)
            self.stat_emb = nn.Embedding(4, cfg.emb_small); d += cfg.emb_small
            d += 1                                                   # t1_same_encounter
            d += 2*len(REC1)                                         # t1 recency value+mask
        # ---- tier 2 ----
        if 2 in T:
            d += add('t2_spec', SPEC_V, cfg.emb_spec)
            d += 3                                                   # pending, time_imprecise, n_pending
        # ---- tier 3 ----
        if 3 in T:
            d += add('t3_org', ORG_V, cfg.emb_org)
            self.rec_emb = nn.Embedding(3, cfg.emb_small); d += cfg.emb_small
            d += 2*len(REC3) + 1                                     # t3 recency value+mask + n_specimens
        # ---- tier 4 ----
        if 4 in T:
            d += 3                                                   # t4 aggregate burden
        self.in_dim = d
        self.mlp = nn.Sequential(
            nn.Linear(d, cfg.d_hidden_enc), nn.LayerNorm(cfg.d_hidden_enc), Act(),
            nn.Dropout(cfg.dropout_enc),
            nn.Linear(cfg.d_hidden_enc, cfg.d_e), nn.LayerNorm(cfg.d_e), Act())

    def _bag(self, key, batch):
        flat, off = batch[key]; m = self.mods[key]
        if self.kinds[key] == 'bag':
            return m(flat.to(next(m.parameters()).device), off.to(next(m.parameters()).device))
        # onehot_mlp: rebuild multi-hot from flat+offsets, then Linear
        B = off.numel(); dev_ = next(m.parameters()).device
        oh = torch.zeros(B, m.in_features, device=dev_)
        # ends in plain Python -- torch.cat([off[1:], tensor([...])]) mixed a cpu scalar
        # with a cuda `off` on GPU and raised a device mismatch (only the D2 onehot path).
        off_l = off.tolist(); ends = off_l[1:] + [int(flat.numel())]
        for b,(s,e) in enumerate(zip(off_l, ends)):
            if e > s: oh[b, flat[s:e].to(dev_)] = 1.0
        return m(oh)

    def forward(self, batch):
        T = self.T; parts = [self.set_emb(batch['set_id']),
                              batch['lab_val'], batch['lab_mask']]
        if self.n_vit:  parts += [batch['vit_val'], batch['vit_mask']]
        if self.n_como: parts += [batch['como']]
        if 1 in T:
            parts += [self._bag('t1_org',batch), self._bag('t1_spec',batch),
                      self.stat_emb(batch['stat_id']),
                      batch['t1_same_encounter'][:,None], batch['rec1_v'], batch['rec1_m']]
        if 2 in T:
            parts += [self._bag('t2_spec',batch), batch['t2_pending'][:,None],
                      batch['t2_time_imprecise'][:,None], batch['t2_n_pending'][:,None]]
        if 3 in T:
            parts += [self._bag('t3_org',batch), self.rec_emb(batch['rec_id']),
                      batch['rec3_v'], batch['rec3_m'], batch['t3_n_specimens'][:,None]]
        if 4 in T:
            parts += [batch['t4_n_prior_positive_ever'][:,None],
                      batch['t4_years_since_first_culture'][:,None],
                      batch['t4_n_prior_encounters'][:,None]]
        return self.mlp(torch.cat(parts, dim=1))

## C5 · FiLM + fusion model (§B6–B7)

`γ,β` are **activations** the generator emits per patient from `e` (not free
parameters). Identity init zeroes the generator's last layer so `γ=1,β=0` at step 0
and fusion starts exactly at CXR-only — the anti-fusion-gap guarantee. `fusion="none"`
shares the same `LayerNorm` so that guarantee is *exact* (see header, design point 1).

In [6]:
class FiLMGenerator(nn.Module):
    def __init__(self, d_e, d_cxr, gamma_mode='full', identity=True):
        super().__init__(); self.gamma_mode = gamma_mode
        self.to_gamma = nn.Linear(d_e, d_cxr); self.to_beta = nn.Linear(d_e, d_cxr)
        if identity:
            for m in (self.to_gamma, self.to_beta): nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, e):
        g = self.to_gamma(e)
        if self.gamma_mode == 'gating_only':
            return torch.sigmoid(g), torch.zeros_like(g)      # (0,1); NOT identity-able
        return 1.0 + torch.tanh(g), self.to_beta(e)           # centered at 1, beta free

class FusionModel(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg
        self.norm = nn.LayerNorm(cfg.d_cxr)                   # shared by none & film (exact identity)
        if cfg.fusion in ('concat','film'):
            self.enc = EhrEncoder(cfg)
        if cfg.fusion == 'film':
            self.gen = FiLMGenerator(cfg.d_e, cfg.d_cxr, cfg.film_gamma_mode, cfg.film_identity_init)
        head_in = cfg.d_cxr
        if cfg.fusion == 'concat':                       head_in = cfg.d_cxr + cfg.d_e
        if cfg.fusion == 'film' and cfg.head_input=='h_concat_e': head_in = cfg.d_cxr + cfg.d_e
        self.head = nn.Sequential(
            nn.Linear(head_in, cfg.d_hidden_head), nn.LayerNorm(cfg.d_hidden_head), Act(),
            nn.Dropout(cfg.dropout_head), nn.Linear(cfg.d_hidden_head, len(LABELS)))
        self._last_ge = None                                  # cache (gamma,beta) for film_reg

    def forward(self, batch):
        x = batch['x']
        if self.cfg.fusion == 'none':
            feat = self.norm(x)
        elif self.cfg.fusion == 'concat':
            feat = torch.cat([self.norm(x), self.enc(batch)], dim=1)
        else:                                                 # film
            e = self.enc(batch); gamma, beta = self.gen(e); self._last_ge = (gamma, beta)
            h = self.norm(gamma * x + beta)
            feat = torch.cat([h, e], dim=1) if self.cfg.head_input=='h_concat_e' else h
        return self.head(feat)

def build_model(cfg): return FusionModel(cfg).to(dev)

## C6 · Losses (§B8) — DB Loss default, `bce_posweight`/`asl` for D5

`ResampleLoss` is the verbatim `wutong16/DistributionBalancedLoss` port (identical to
04b·C3): rebalanced weighting + negative-tolerant regularization, `class_freq` from 02.
`bce_posweight` and `asl` are the D5 A/B baselines; `ral` (Robust Asymmetric Loss) is a
TODO stub per §B8.

In [7]:
class ResampleLoss(nn.Module):
    """DB Loss (Wu et al., ECCV 2020), wutong16/DistributionBalancedLoss. Math verbatim."""
    def __init__(self, class_freq, train_num, reduction='mean',
                 focal=dict(focal=True, alpha=0.5, gamma=2.0),
                 map_param=dict(alpha=0.1, beta=10.0, gamma=0.9),
                 logit_reg=dict(neg_scale=2.0, init_bias=0.05)):
        super().__init__()
        self.reduction = reduction
        self.focal=focal['focal']; self.gamma=focal['gamma']; self.alpha=focal['alpha']
        self.map_alpha,self.map_beta,self.map_gamma = map_param['alpha'],map_param['beta'],map_param['gamma']
        cf = torch.as_tensor(np.asarray(class_freq), dtype=torch.float)
        self.train_num = float(train_num)
        self.register_buffer('freq_inv', torch.ones_like(cf)/cf)
        self.neg_scale = logit_reg.get('neg_scale', 1.0)
        ib = logit_reg.get('init_bias', 0.0)
        self.register_buffer('init_bias', -torch.log(self.train_num/cf - 1)*ib/self.neg_scale)
        self.logit_reg = logit_reg
    def rebalance_weight(self, gt):
        rr = torch.sum(gt*self.freq_inv, dim=1, keepdim=True)
        pw = self.freq_inv.unsqueeze(0)/rr
        return torch.sigmoid(self.map_beta*(pw - self.map_gamma)) + self.map_alpha
    def logit_reg_functions(self, labels, logits, weight):
        if 'init_bias' in self.logit_reg: logits = logits + self.init_bias
        if 'neg_scale' in self.logit_reg:
            logits = logits*(1-labels)*self.neg_scale + logits*labels
            weight = weight/self.neg_scale*(1-labels) + weight*labels
        return logits, weight
    def forward(self, cls_score, label):
        label = label.float(); weight = self.rebalance_weight(label)
        cls_score, weight = self.logit_reg_functions(label, cls_score, weight)
        if self.focal:
            logpt = F.binary_cross_entropy_with_logits(cls_score, label, reduction='none')
            pt = torch.exp(-logpt)
            wt = F.binary_cross_entropy_with_logits(cls_score, label, weight=weight, reduction='none')
            at = torch.where(label==1, self.alpha, 1-self.alpha)
            loss = at*((1-pt)**self.gamma)*wt
        else:
            loss = F.binary_cross_entropy_with_logits(cls_score, label, weight=weight, reduction='none')
        return loss.mean() if self.reduction=='mean' else loss.sum()

class ASLoss(nn.Module):
    """Asymmetric Loss (Ben-Baruch et al. 2021). Ported from Alibaba-MIIL/ASL, math verbatim.

    An earlier hand-rolled version of this collapsed D5_asl to 0.2778 (-0.126). The bug:
    it used `p*(1-y)` -- the POSITIVE probability masked to negatives -- where the paper
    uses `xs_neg = 1-p`. That inverts pt on the negative branch, so easy negatives got a
    large loss and hard negatives got exactly zero: backwards from focal logic, and since
    negatives are 67-99% of every column the gradient drowned. Positives were unaffected,
    which is why it looked plausible. §B8 says port, do not re-derive -- this is why.
    """
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg, self.gamma_pos, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps
    def forward(self, x, y):
        y = y.float()
        xs_pos = torch.sigmoid(x)
        xs_neg = 1.0 - xs_pos
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)     # asymmetric clip: drop easy/mislabeled negs
        los = y*torch.log(xs_pos.clamp(min=self.eps)) + (1-y)*torch.log(xs_neg.clamp(min=self.eps))
        pt = xs_pos*y + xs_neg*(1-y)                       # prob of the TRUE class (1-p for negatives)
        g  = self.gamma_pos*y + self.gamma_neg*(1-y)
        return -(los*torch.pow(1-pt, g)).mean()

# regression check for the bug above: an easy negative must contribute ~0, a hard one must not.
with torch.no_grad():
    _a = ASLoss()
    _easy = _a(torch.tensor([[-4.6]]), torch.tensor([[0.0]])).item()   # p~0.01, label 0
    _hard = _a(torch.tensor([[ 4.6]]), torch.tensor([[0.0]])).item()   # p~0.99, label 0
assert _easy < 1e-3 and _hard > 1.0, (_easy, _hard)
print(f'ASL check: easy-negative loss {_easy:.5f} (~0 OK) | hard-negative loss {_hard:.3f} (large OK)')

def build_loss(cfg):
    if cfg.loss == 'db':
        return ResampleLoss(CLASS_FREQ, NUM_TRAIN).to(dev)
    if cfg.loss == 'asl':
        return ASLoss().to(dev)
    if cfg.loss == 'bce_posweight':
        pw = torch.as_tensor((NUM_TRAIN - CLASS_FREQ)/np.clip(CLASS_FREQ,1,None), dtype=torch.float, device=dev)
        return lambda s,y: F.binary_cross_entropy_with_logits(s, y.float(), pos_weight=pw)
    # if cfg.loss == 'ral':  # TODO(B8): Robust Asymmetric Loss -- beat ASL on CXR-LT, noise-robust
    raise ValueError(cfg.loss)

ASL check: easy-negative loss -0.00000 (~0 OK) | hard-negative loss 2.198 (large OK)


## C7 · §E2 model gates — run before any training

The five §E2 checks, most importantly **E2.1 (identity start)**: with `film` + `full`
+ `identity_init`, the model's logits must equal the CXR-only path `head(norm(x))` for
*any* `e` — because a zeroed generator emits `γ=1,β=0` regardless of input. If this
fails, the anti-fusion-gap guarantee is broken and every FiLM result is suspect.

In [8]:
def a_batch(n=8):
    b = collate([FusionDS(idx_tr)[k] for k in range(n)])
    return {k:(tuple(t.to(dev) for t in v) if isinstance(v,tuple) else v.to(dev))
            for k,v in b.items()}

# E2.2 shapes -- every fusion mode -> [B,13]
for fu in ['none','concat','film']:
    m = build_model(replace(BASE, fusion=fu)); m.eval()
    with torch.no_grad(): o = m(a_batch())
    assert o.shape == (8, 13), (fu, o.shape); print(f'E2.2 shapes {fu:6s} -> {tuple(o.shape)}  OK')

# E2.1 identity start -- film@init == head(norm(x)), independent of e
m = build_model(replace(BASE, fusion='film', film_gamma_mode='full', film_identity_init=True)); m.eval()
with torch.no_grad():
    b = a_batch(); ref = m.head(m.norm(b['x']))             # the CXR-only path
    got = m(b)
    assert torch.allclose(ref, got, atol=1e-5), (ref-got).abs().max().item()
print('E2.1 identity start: FiLM logits == head(norm(x))  OK  (anti-fusion-gap gate)')

# E2.3 EmbeddingBag multi-value -- 3 ids and 0 ids both give one row of d_e
enc = EhrEncoder(BASE).to(dev); enc.eval()
b3 = collate([FusionDS(idx_tr)[k] for k in range(3)])
b3 = {k:(tuple(t.to(dev) for t in v) if isinstance(v,tuple) else v.to(dev)) for k,v in b3.items()}
with torch.no_grad(): e = enc(b3)
assert e.shape == (3, BASE.d_e); print(f'E2.3 EmbeddingBag ragged -> e {tuple(e.shape)}  OK')

# E2.4 DB loss finite + rarer class gets a larger rebalance weight.
# Probe correctly: within ONE sample, pw_j = freq_inv[j]/repeat_rate ~ 1/n_j, so a
# rarer class j is up-weighted RELATIVE to a common one. (The diagonal of
# rebalance_weight(eye) is constant by construction -- a single-label sample normalizes
# its own positive to pw=1 -- so it cannot show the effect; use a fixed reference row.)
rare, common = int(TRAIN_PREV.argmin()), int(TRAIN_PREV.argmax())
dbl = ResampleLoss(CLASS_FREQ, NUM_TRAIN).to(dev)
with torch.no_grad():
    l = dbl(torch.randn(16,13,device=dev), torch.zeros(16,13,device=dev)); assert torch.isfinite(l)
    gt = torch.zeros(1,13,device=dev); gt[0, common] = 1.0      # a sample positive in the common class
    w  = dbl.rebalance_weight(gt)[0].cpu().numpy()              # per-class weight for that sample
assert w[rare] > w[common], (w[rare], w[common])
print(f'E2.4 DB loss finite ({l.item():.3f}); within-sample rebalance '
      f'w[{LABELS[rare]}]={w[rare]:.3f} > w[{LABELS[common]}]={w[common]:.3f}  OK')

# E2.5 overfit one batch -> loss collapses
m = build_model(replace(BASE, fusion='film')); opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
b = a_batch(64); crit = ResampleLoss(CLASS_FREQ, NUM_TRAIN).to(dev)
l0=None
for _ in range(200):
    opt.zero_grad(); l = crit(m(b), b['y']); l.backward(); opt.step()
    l0 = l0 or l.item()
print(f'E2.5 overfit-one-batch: {l0:.3f} -> {l.item():.4f}  (must fall)')

E2.2 shapes none   -> (8, 13)  OK
E2.2 shapes concat -> (8, 13)  OK
E2.2 shapes film   -> (8, 13)  OK
E2.1 identity start: FiLM logits == head(norm(x))  OK  (anti-fusion-gap gate)
E2.3 EmbeddingBag ragged -> e (3, 128)  OK
E2.4 DB loss finite (0.163); within-sample rebalance w[Pleural_Other]=1.100 > w[No_Finding]=0.831  OK
E2.5 overfit-one-batch: 0.058 -> 0.0000  (must fall)


## C8 · Metrics (§C) — macro AUPRC primary, thresholds on val, confound-corrected Δ

- **Primary macro AUPRC**; AUROC auxiliary (inflated at <2% prevalence).
- Per-label **thresholds tuned on val** (max F1), reporting **precision AND recall** at
  the operating point — never a global 0.5.
- **Confound-corrected Δ** for complementarity: naive `fusion−cxr` is confounded with
  the CXR level, so we also report headroom-normalized `Δ/(1−AP_cxr)` and stratify Δ by
  `setting` (fusion must not be learning an inpatient/ED shortcut).

In [9]:
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

@torch.no_grad()
def predict(model, idx, bs=4096):
    model.eval(); P = np.zeros((len(idx), len(LABELS)), 'float32')
    dl = loader(idx, False, bs); pos = 0
    for b in dl:
        b = {k:(tuple(t.to(dev) for t in v) if isinstance(v,tuple) else v.to(dev)) for k,v in b.items()}
        p = torch.sigmoid(model(b)).float().cpu().numpy(); P[pos:pos+len(p)] = p; pos += len(p)
    return P

def per_label_ap(Yt, P):
    return np.array([average_precision_score(Yt[:,j], P[:,j]) for j in range(len(LABELS))])

def tune_thresholds(Yv, Pv):
    th = np.zeros(len(LABELS))
    for j in range(len(LABELS)):
        pr, rc, t = precision_recall_curve(Yv[:,j], Pv[:,j])
        f1 = 2*pr*rc/(pr+rc+1e-9); th[j] = t[max(f1[:-1].argmax(),0)] if len(t) else 0.5
    return th

def report(model, tag):
    Pv, Pte = predict(model, idx_va), predict(model, idx_te)
    Yv, Yte = Y[idx_va], Y[idx_te]
    ap = per_label_ap(Yte, Pte)
    auc = np.array([roc_auc_score(Yte[:,j], Pte[:,j]) if len(np.unique(Yte[:,j]))>1 else np.nan
                    for j in range(len(LABELS))])
    th = tune_thresholds(Yv, Pv)
    prec = np.zeros(len(LABELS)); rec = np.zeros(len(LABELS))
    for j in range(len(LABELS)):
        pred = (Pte[:,j] >= th[j]); tp = (pred & (Yte[:,j]==1)).sum()
        prec[j] = tp/max(pred.sum(),1); rec[j] = tp/max((Yte[:,j]==1).sum(),1)
    r = pd.DataFrame({'label':LABELS,'tier':TIER,'prev':TRAIN_PREV.round(4),
                      'AUPRC':ap.round(4),'AUROC':auc.round(4),
                      'precision@F1':prec.round(3),'recall@F1':rec.round(3)}).sort_values('prev')
    print(f'\n=== {tag} ===  macro mAP {ap.mean():.4f} | macro AUROC {np.nanmean(auc):.4f}')
    print(r.to_string(index=False))
    print('by tier:'); print(r.groupby('tier')[['AUPRC','AUROC','precision@F1','recall@F1']].mean().round(4).to_string())
    return dict(tag=tag, macro_mAP=float(ap.mean()), macro_AUROC=float(np.nanmean(auc)),
                ap=ap, auc=auc, th=th, P_test=Pte)

def confound_delta(res_cxr, res_fusion):
    """Δ per label with the CXR level divided out (headroom-normalized), + setting strat."""
    d = res_fusion['ap'] - res_cxr['ap']
    dn = d / np.clip(1 - res_cxr['ap'], 1e-3, None)
    tbl = pd.DataFrame({'label':LABELS,'tier':TIER,'AP_cxr':res_cxr['ap'].round(4),
                        'AP_fusion':res_fusion['ap'].round(4),'delta':d.round(4),
                        'delta_norm':dn.round(4)}).sort_values('delta', ascending=False)
    print(f'\n--- complementarity Δ  ({res_fusion["tag"]} vs {res_cxr["tag"]}) ---')
    print(tbl.to_string(index=False))
    tail_d = f'{d[TIER=="tail"].mean():+.4f}' if (TIER=="tail").any() else 'n/a'
    print(f'macro Δ {d.mean():+.4f} | headroom-norm Δ {dn.mean():+.4f} | tail Δ {tail_d}')
    # setting stratification: does the gain survive within each setting?
    print('Δ macro-mAP within each setting (guards against a setting shortcut):')
    setl = ehr['setting'].astype(str).str.lower().to_numpy()[idx_te]   # 'ed' is lowercase
    for s in ['inpatient','ed','outpatient']:
        m = (setl == s)
        if m.sum() < 50: continue
        a_c = per_label_ap(Y[idx_te][m], res_cxr['P_test'][m]).mean()
        a_f = per_label_ap(Y[idx_te][m], res_fusion['P_test'][m]).mean()
        print(f'   {s:11s} n={int(m.sum()):6,d}  cxr {a_c:.4f} -> fusion {a_f:.4f}  Δ {a_f-a_c:+.4f}')
    return tbl

## C9 · Train one config — AdamW + cosine-with-warmup, resume, best-val-mAP

Param groups per §C: weight decay on the weights, **zero** on norms and biases.
Checkpoints on best val macro AUPRC; `*_last.pt` every epoch for resume (LR schedule
state included, same discipline as 04b). Config-tagged so a sweep never overwrites.

**Early stopping** (`cfg.patience`): this head peaks early (~ep5) then overfits — train
loss keeps falling while val mAP declines, the textbook signature. `best` is immutable
so the *result* is safe either way, but without a stop each config would burn ~20 idle
epochs, and the D1–D5 grid multiplies that. Stop after `patience` epochs with no val
gain; `epochs` is just the hard cap. The heavier lever against the overfit itself
(more `wd`/`dropout`) is left for Tuan to tune.

In [10]:
def param_groups(model, wd):
    decay, no_decay = [], []
    for n,p in model.named_parameters():
        if not p.requires_grad: continue
        (no_decay if p.ndim <= 1 else decay).append(p)     # norms/biases -> no decay
    return [{'params':decay,'weight_decay':wd},{'params':no_decay,'weight_decay':0.0}]

def cosine_warmup(opt, total, warmup):
    def f(step):
        if step < warmup: return step/max(warmup,1)
        p = (step-warmup)/max(total-warmup,1); return 0.5*(1+math.cos(math.pi*p))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def ckpt(tag, kind): return os.path.join(CKPT_DIR, f'fusion_{tag}_{kind}.pt')
def save_atomic(path, obj): torch.save(obj, path+'.tmp'); os.replace(path+'.tmp', path)

def run_config(cfg, verbose=True):
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)   # per-config seed -> C12 error bars
    model = build_model(cfg); crit = build_loss(cfg)
    opt = torch.optim.AdamW(param_groups(model, cfg.wd), lr=cfg.lr)
    dl = loader(idx_tr, True, cfg.batch)
    total = cfg.epochs*len(dl); sch = cosine_warmup(opt, total, int(cfg.warmup_frac*total))
    start, best, since = 0, -1.0, 0
    if os.path.exists(ckpt(cfg.tag,'last')):
        c = torch.load(ckpt(cfg.tag,'last'), map_location=dev, weights_only=False)
        model.load_state_dict(c['model']); opt.load_state_dict(c['opt']); sch.load_state_dict(c['sch'])
        start, best, since = c['epoch']+1, c['best'], c.get('since', 0)
        if verbose: print(f"resumed {cfg.tag} @ep{start} (best {best:.4f}, {since} since improve)")
    # already done, OR early-stopped in a prior run (patience already exhausted)
    done = start >= cfg.epochs or (cfg.patience and since >= cfg.patience)
    if done and os.path.exists(ckpt(cfg.tag,'best')):
        model.load_state_dict(torch.load(ckpt(cfg.tag,'best'), map_location=dev, weights_only=False)['model'])
        return model, best
    for ep in range(start, cfg.epochs):
        model.train(); tot = 0
        it = tqdm(dl, desc=f'{cfg.tag} ep{ep+1}/{cfg.epochs}', leave=False) if verbose else dl
        for b in it:
            b = {k:(tuple(t.to(dev) for t in v) if isinstance(v,tuple) else v.to(dev)) for k,v in b.items()}
            opt.zero_grad(); logits = model(b); loss = crit(logits, b['y'])
            if cfg.fusion=='film' and cfg.film_reg>0 and model._last_ge is not None:
                g,be = model._last_ge; loss = loss + cfg.film_reg*((g-1).pow(2).mean()+be.pow(2).mean())
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step(); sch.step(); tot += loss.item()
        Pv = predict(model, idx_va); mAP = per_label_ap(Y[idx_va], Pv).mean()
        improved = mAP > best
        if improved:
            best = mAP; since = 0
            save_atomic(ckpt(cfg.tag,'best'), dict(model=model.state_dict(), cfg=vars(cfg), val_mAP=mAP, epoch=ep))
        else:
            since += 1
        save_atomic(ckpt(cfg.tag,'last'), dict(model=model.state_dict(), opt=opt.state_dict(),
                    sch=sch.state_dict(), epoch=ep, best=best, since=since))
        if verbose: print(f'  {cfg.tag} ep{ep+1}: train {tot/len(dl):.4f} | val mAP {mAP:.4f}'
                          + ('  *' if improved else f'  ({since}/{cfg.patience})'))
        if cfg.patience and since >= cfg.patience:
            if verbose: print(f'  early stop: no val gain in {cfg.patience} epochs (best {best:.4f})')
            break
    model.load_state_dict(torch.load(ckpt(cfg.tag,'best'), map_location=dev, weights_only=False)['model'])
    return model, best

## C10 · Baselines first (§F step 11) — `none → concat → film`

The three primary configs, trained in order. `none` is the CXR-only reference every Δ
is measured against; `concat` is the naive fusion the plan flags as underperforming;
`film` is the candidate. Then the confound-corrected Δ tables (§C).

In [11]:
results = {}
for fu in ['none','concat','film']:
    cfg = replace(BASE, fusion=fu, tag=f'A0_{fu}' if fu=='none' else f'A_{fu}')
    print(f'\n########## training fusion={fu} ({cfg.tag}) ##########')
    model, best = run_config(cfg)
    results[fu] = report(model, cfg.tag)

print('\n\n================= SUMMARY (test) =================')
sm = pd.DataFrame([{'config':results[f]['tag'],'macro_mAP':round(results[f]['macro_mAP'],4),
                    'macro_AUROC':round(results[f]['macro_AUROC'],4)} for f in ['none','concat','film']])
print(sm.to_string(index=False))

confound_delta(results['none'], results['concat'])
confound_delta(results['none'], results['film'])
print('\nRead §C: judge complementarity on per-label AUPRC + headroom-norm Δ + setting '
      'stratification, NOT on macro F1 or naive fusion-minus-cxr.')


########## training fusion=none (A0_none) ##########
resumed A0_none @ep10 (best 0.3945, 6 since improve)

=== A0_none ===  macro mAP 0.3909 | macro AUROC 0.7938
                     label   tier   prev  AUPRC  AUROC  precision@F1  recall@F1
             Pleural_Other   tail 0.0119 0.0626 0.7913         0.089      0.179
                  Fracture medium 0.0217 0.0715 0.7102         0.098      0.144
               Lung_Lesion medium 0.0322 0.2033 0.7824         0.231      0.328
              Pneumothorax medium 0.0511 0.4537 0.8810         0.464      0.477
             Consolidation medium 0.0672 0.1757 0.7616         0.174      0.470
Enlarged_Cardiomediastinum medium 0.0734 0.1367 0.6888         0.128      0.473
                 Pneumonia   head 0.1529 0.3867 0.7503         0.336      0.537
                     Edema   head 0.1797 0.6068 0.8743         0.506      0.704
              Cardiomegaly   head 0.2254 0.4910 0.7883         0.425      0.686
              Lung_Opacity   head 0.2

## C11 · Ablation grid (§D1–D5) — parameterized, default off

Each list below is one ablation axis from Part D. Flip `RUN_ABLATIONS=True` to sweep;
because every config is cached-feature-only, the whole grid is minutes, not hours. Each
run is checkpoint-tagged, so an interrupted sweep resumes where it stopped.

In [12]:
22RUN_ABLATIONS = True          # set True to run D1-D5

GRID = {
 'D1_arch'   : [replace(BASE, fusion='film', film_identity_init=idn, tag=f'D1_id{int(idn)}') for idn in (True,False)]
             + [replace(BASE, fusion='film', film_gamma_mode='gating_only', tag='D1_gate')]
             + [replace(BASE, fusion='film', head_input='h_concat_e', tag='D1_hcat')],
 'D3_tiers'  : [replace(BASE, fusion='film', micro_tiers=t, tag=f'D3_{"".join(map(str,t)) or "labs"}')
                for t in [(), (1,2), (1,2,3), (1,2,3,4)]],
 'D4_mech'   : [replace(BASE, fusion='film', micro_tiers=(1,2,3),      tag='D4_t3_90d'),
                replace(BASE, fusion='film', micro_tiers=(1,2,3,4),    tag='D4_t3_90d_t4')],
 'D5_loss'   : [replace(BASE, fusion='film', loss=l, tag=f'D5_{l}') for l in ('db','asl','bce_posweight')],
 'D2_encode' : [replace(BASE, fusion='film', ehr_cat_encoding=e, tag=f'D2_{e}')
                for e in ('embedding','onehot_mlp')],
 # D6/D7 are not in the plan -- they test the blocks added after it (02 C7-bis/C7-ter).
 # D7 is the one that matters: if como helps, is it the diagnosis, or just "this patient
 # gets admitted a lot" leaking through the report-writing channel (A4)? Compare it
 # against t4 utilisation, which D4 already showed adds ~nothing.
 'D6_vitals' : [replace(BASE, fusion='film', use_vitals=v, tag=f'D6_vit{int(v)}') for v in (True, False)],
 'D7_como'   : [replace(BASE, fusion='film', use_como=c,   tag=f'D7_como{int(c)}') for c in (True, False)],
}

if RUN_ABLATIONS:
    abl = {}
    for axis, cfgs in GRID.items():
        print(f'\n===== {axis} =====')
        for cfg in cfgs:
            model, best = run_config(cfg, verbose=False)
            r = report(model, cfg.tag); abl[cfg.tag] = r['macro_mAP']
    print('\n===== ablation macro-mAP summary =====')
    for k,v in abl.items(): print(f'  {k:16s} {v:.4f}')
else:
    print('RUN_ABLATIONS=False -> baselines only. Set True in this cell to sweep D1-D5.')
    print('grid sizes:', {k:len(v) for k,v in GRID.items()})
print('\nDONE. Fusion pipeline 01->06 complete.')


===== D1_arch =====

=== D1_id1 ===  macro mAP 0.4037 | macro AUROC 0.8102
                     label   tier   prev  AUPRC  AUROC  precision@F1  recall@F1
             Pleural_Other   tail 0.0119 0.0760 0.8266         0.107      0.195
                  Fracture medium 0.0217 0.0844 0.7472         0.116      0.219
               Lung_Lesion medium 0.0322 0.2126 0.7994         0.217      0.363
              Pneumothorax medium 0.0511 0.4585 0.8923         0.511      0.447
             Consolidation medium 0.0672 0.1899 0.7792         0.180      0.474
Enlarged_Cardiomediastinum medium 0.0734 0.1476 0.7238         0.140      0.539
                 Pneumonia   head 0.1529 0.4088 0.7663         0.377      0.495
                     Edema   head 0.1797 0.6257 0.8828         0.524      0.704
              Cardiomegaly   head 0.2254 0.5133 0.8035         0.437      0.710
              Lung_Opacity   head 0.2448 0.4601 0.7503         0.396      0.731
               Atelectasis   head 0.2502 0.5

## C12 · Seed sweep — error bars, because single runs cannot settle this

Every claim so far rests on one run per config, and the margins are tiny: `film` vs
`concat` **flipped** when the LR schedule changed (40-ep: film 0.4051 > concat 0.4022;
25-ep: concat 0.4049 > film 0.4032). A difference that reverses under a nuisance change
is not a result. **Nothing below ~0.005 is claimable without a spread**, so this cell
runs each config over several seeds and reports mean ± 95% CI.

Two questions it settles:

1. **Which of `none / concat / film / film+bce` are actually distinguishable?**
2. **Is the "micro is Pneumonia-specific" story real?** `D3_labs` vs `D3_12` differed by
   `+0.0156` on Pneumonia in one run — but that label was picked *after* seeing the
   result, out of 13, so it is exactly the shape a multiple-comparisons artifact takes.
   Per-label CIs across seeds either survive it or kill it.

`film_db` is the same config as `A_film` and `D5_db` (BASE is already `fusion='film',
loss='db'`); it appears once here, not three times.

**Honesty about power:** 3 seeds gives 2 degrees of freedom, so the 95% CI uses
`t=4.30` and is wide. That is the point — it reports how little a 3-seed study can
resolve, rather than hiding it. Raise `SEEDS` if a margin lands near the boundary.

In [13]:
from scipy import stats as sstats

RUN_SEEDS = True
SEEDS     = [42, 1337, 2024]          # 3 seeds x 6 configs = 18 runs, ~20-35 min on GPU

SWEEP = {
    'none'    : replace(BASE, fusion='none'),
    'concat'  : replace(BASE, fusion='concat'),
    'film_db' : replace(BASE, fusion='film', loss='db'),             # == A_film == D5_db
    'film_bce': replace(BASE, fusion='film', loss='bce_posweight'),
    # asl only became measurable once the port was fixed (C6) -- it then won the whole
    # grid at 0.4192 on one seed, so it is the config most in need of error bars.
    'film_asl': replace(BASE, fusion='film', loss='asl'),
    'D3_labs' : replace(BASE, fusion='film', micro_tiers=()),        # labs only
    'D3_12'   : replace(BASE, fusion='film', micro_tiers=(1,2)),     # + fresh/pending culture
}

@torch.no_grad()
def eval_ap(model):
    return per_label_ap(Y[idx_te], predict(model, idx_te))           # [13] test AP, no printing

def ci95(v):
    v = np.asarray(v, float); n = len(v)
    if n < 2: return float(v.mean()), 0.0
    return float(v.mean()), float(sstats.t.ppf(0.975, n-1) * v.std(ddof=1) / np.sqrt(n))

if RUN_SEEDS:
    AP = {k: [] for k in SWEEP}                                      # name -> [ap[13] per seed]
    for name, base_cfg in SWEEP.items():
        for s in SEEDS:
            cfg = replace(base_cfg, seed=s, tag=f'S_{name}_s{s}')
            model, _ = run_config(cfg, verbose=False)
            AP[name].append(eval_ap(model))
        m = [a.mean() for a in AP[name]]
        print(f'  {name:9s} seeds {[round(x,4) for x in m]}  mean {np.mean(m):.4f}')

    # ---------- 1. macro mAP with error bars ----------
    print('\n===== macro mAP over seeds (mean +/- 95% CI) =====')
    rows = []
    for name in SWEEP:
        mu, h = ci95([a.mean() for a in AP[name]])
        rows.append(dict(config=name, mean=round(mu,4), ci95=round(h,4),
                         lo=round(mu-h,4), hi=round(mu+h,4), n=len(AP[name])))
    print(pd.DataFrame(rows).to_string(index=False))

    # ---------- 2. pairwise: is the gap bigger than the noise? ----------
    print('\n===== pairwise (Welch t on seed means; n=3 -> low power by construction) =====')
    def cmp(a, b):
        ma = np.array([x.mean() for x in AP[a]]); mb = np.array([x.mean() for x in AP[b]])
        d = mb.mean() - ma.mean()
        se = np.sqrt(ma.var(ddof=1)/len(ma) + mb.var(ddof=1)/len(mb))
        t, p = sstats.ttest_ind(mb, ma, equal_var=False)
        verdict = 'distinguishable' if p < 0.05 else 'NOT distinguishable'
        print(f'  {b:9s} - {a:9s} = {d:+.4f}  +/-{1.96*se:.4f}   p={p:.3f}   {verdict}')
    for a, b in [('none','concat'), ('none','film_db'), ('concat','film_db'),
                 ('film_db','film_bce'), ('film_db','film_asl'), ('film_bce','film_asl'),
                 ('D3_labs','D3_12')]:
        cmp(a, b)

    # ---- 2b. loss x tier: macro hides the trade the losses actually make ----
    # DB Loss exists FOR the tail. A single seed said bce beat it on macro (+0.012) while
    # LOSING the tail (0.0779 vs 0.0826). Macro alone cannot adjudicate that, so compare
    # the three losses per tier, with the spread.
    print('\n===== loss x tier (mean +/- 95% CI over seeds) -- where each loss pays =====')
    rows = []
    for name in ('film_db', 'film_bce', 'film_asl'):
        if name not in AP: continue
        r = {'loss': name.replace('film_', '')}
        for t in ('head', 'medium', 'tail'):
            m = [a[TIER == t].mean() for a in AP[name]]
            mu, h = ci95(m); r[t] = f'{mu:.4f}+/-{h:.4f}'
        rows.append(r)
    print(pd.DataFrame(rows).to_string(index=False))
    print('  a loss that wins macro but loses tail is a different research answer than one')
    print('  that wins both -- the plan chose db BECAUSE of the tail (B8).')

    # ---------- 3. the Pneumonia question, per label, with CIs ----------
    print('\n===== D3_12 - D3_labs, PER LABEL (does micro help, and where?) =====')
    out = []
    for j, l in enumerate(LABELS):
        a = np.array([x[j] for x in AP['D3_labs']]); b = np.array([x[j] for x in AP['D3_12']])
        d = b.mean() - a.mean()
        se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
        _, p = sstats.ttest_ind(b, a, equal_var=False)
        out.append(dict(label=l, tier=TIER[j], labs=round(a.mean(),4), plus_micro=round(b.mean(),4),
                        delta=round(d,4), ci95=round(1.96*se,4), p=round(p,3),
                        sig='*' if p < 0.05 else ''))
    t = pd.DataFrame(out).sort_values('delta', ascending=False)
    print(t.to_string(index=False))
    n_sig = int((t.p < 0.05).sum())
    print(f'\nlabels where micro moves AP at p<0.05: {n_sig}/13'
          f'   (Bonferroni over 13 labels wants p<{0.05/13:.4f})')
    pn = t[t.label == 'Pneumonia'].iloc[0]
    print(f'Pneumonia: {pn.labs:.4f} -> {pn.plus_micro:.4f}  delta {pn.delta:+.4f} '
          f'+/-{pn.ci95:.4f}  p={pn.p:.3f}')
    print('  -> the single-run +0.0156 survives only if this CI excludes 0 AND p holds up')
    print('     against the 13-label Bonferroni bar; otherwise it was multiple comparisons.')
else:
    print('RUN_SEEDS=False. Set True to get error bars.')
    print(f'cost: {len(SWEEP)} configs x {len(SEEDS)} seeds = {len(SWEEP)*len(SEEDS)} runs')

  none      seeds [np.float64(0.391), np.float64(0.391), np.float64(0.3927)]  mean 0.3916
  concat    seeds [np.float64(0.4029), np.float64(0.4046), np.float64(0.4051)]  mean 0.4042
  film_db   seeds [np.float64(0.4037), np.float64(0.4079), np.float64(0.406)]  mean 0.4059
  film_bce  seeds [np.float64(0.4151), np.float64(0.4156), np.float64(0.415)]  mean 0.4152
  film_asl  seeds [np.float64(0.4192), np.float64(0.4194), np.float64(0.4221)]  mean 0.4202
  D3_labs   seeds [np.float64(0.4047), np.float64(0.4057), np.float64(0.403)]  mean 0.4045
  D3_12     seeds [np.float64(0.4039), np.float64(0.4031), np.float64(0.406)]  mean 0.4043

===== macro mAP over seeds (mean +/- 95% CI) =====
  config   mean   ci95     lo     hi  n
    none 0.3916 0.0025 0.3891 0.3941  3
  concat 0.4042 0.0028 0.4014 0.4070  3
 film_db 0.4059 0.0051 0.4007 0.4110  3
film_bce 0.4152 0.0008 0.4144 0.4161  3
film_asl 0.4202 0.0041 0.4162 0.4243  3
 D3_labs 0.4045 0.0034 0.4011 0.4079  3
   D3_12 0.4043 0.0037 0.4006 

In [14]:
from scipy import stats as sstats

RUN_SEEDS = True
SEEDS     = [42, 1337, 2024]          # 3 seeds x 6 configs = 18 runs, ~20-35 min on GPU

SWEEP = {
    'none'    : replace(BASE, fusion='none'),
    'concat'  : replace(BASE, fusion='concat'),
    'film_db' : replace(BASE, fusion='film', loss='db'),             # == A_film == D5_db
    'film_bce': replace(BASE, fusion='film', loss='bce_posweight'),
    # asl only became measurable once the port was fixed (C6) -- it then won the whole
    # grid at 0.4192 on one seed, so it is the config most in need of error bars.
    'film_asl': replace(BASE, fusion='film', loss='asl'),
    'D3_labs' : replace(BASE, fusion='film', micro_tiers=()),        # labs only
    'D3_12'   : replace(BASE, fusion='film', micro_tiers=(1,2)),     # + fresh/pending culture
}

@torch.no_grad()
def eval_ap(model):
    return per_label_ap(Y[idx_te], predict(model, idx_te))           # [13] test AP, no printing

def ci95(v):
    v = np.asarray(v, float); n = len(v)
    if n < 2: return float(v.mean()), 0.0
    return float(v.mean()), float(sstats.t.ppf(0.975, n-1) * v.std(ddof=1) / np.sqrt(n))

if RUN_SEEDS:
    AP = {k: [] for k in SWEEP}                                      # name -> [ap[13] per seed]
    for name, base_cfg in SWEEP.items():
        for s in SEEDS:
            cfg = replace(base_cfg, seed=s, tag=f'S_{name}_s{s}')
            model, _ = run_config(cfg, verbose=False)
            AP[name].append(eval_ap(model))
        m = [a.mean() for a in AP[name]]
        print(f'  {name:9s} seeds {[round(x,4) for x in m]}  mean {np.mean(m):.4f}')

    # ---------- 1. macro mAP with error bars ----------
    print('\n===== macro mAP over seeds (mean +/- 95% CI) =====')
    rows = []
    for name in SWEEP:
        mu, h = ci95([a.mean() for a in AP[name]])
        rows.append(dict(config=name, mean=round(mu,4), ci95=round(h,4),
                         lo=round(mu-h,4), hi=round(mu+h,4), n=len(AP[name])))
    print(pd.DataFrame(rows).to_string(index=False))

    # ---------- 2. pairwise: is the gap bigger than the noise? ----------
    print('\n===== pairwise (Welch t on seed means; n=3 -> low power by construction) =====')
    def cmp(a, b):
        ma = np.array([x.mean() for x in AP[a]]); mb = np.array([x.mean() for x in AP[b]])
        d = mb.mean() - ma.mean()
        se = np.sqrt(ma.var(ddof=1)/len(ma) + mb.var(ddof=1)/len(mb))
        t, p = sstats.ttest_ind(mb, ma, equal_var=False)
        verdict = 'distinguishable' if p < 0.05 else 'NOT distinguishable'
        print(f'  {b:9s} - {a:9s} = {d:+.4f}  +/-{1.96*se:.4f}   p={p:.3f}   {verdict}')
    for a, b in [('none','concat'), ('none','film_db'), ('concat','film_db'),
                 ('film_db','film_bce'), ('film_db','film_asl'), ('film_bce','film_asl'),
                 ('D3_labs','D3_12')]:
        cmp(a, b)

    # ---- 2b. loss x tier: macro hides the trade the losses actually make ----
    # DB Loss exists FOR the tail. A single seed said bce beat it on macro (+0.012) while
    # LOSING the tail (0.0779 vs 0.0826). Macro alone cannot adjudicate that, so compare
    # the three losses per tier, with the spread.
    print('\n===== loss x tier (mean +/- 95% CI over seeds) -- where each loss pays =====')
    rows = []
    for name in ('film_db', 'film_bce', 'film_asl'):
        if name not in AP: continue
        r = {'loss': name.replace('film_', '')}
        for t in ('head', 'medium', 'tail'):
            m = [a[TIER == t].mean() for a in AP[name]]
            mu, h = ci95(m); r[t] = f'{mu:.4f}+/-{h:.4f}'
        rows.append(r)
    print(pd.DataFrame(rows).to_string(index=False))
    print('  a loss that wins macro but loses tail is a different research answer than one')
    print('  that wins both -- the plan chose db BECAUSE of the tail (B8).')

    # ---------- 3. the Pneumonia question, per label, with CIs ----------
    print('\n===== D3_12 - D3_labs, PER LABEL (does micro help, and where?) =====')
    out = []
    for j, l in enumerate(LABELS):
        a = np.array([x[j] for x in AP['D3_labs']]); b = np.array([x[j] for x in AP['D3_12']])
        d = b.mean() - a.mean()
        se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
        _, p = sstats.ttest_ind(b, a, equal_var=False)
        out.append(dict(label=l, tier=TIER[j], labs=round(a.mean(),4), plus_micro=round(b.mean(),4),
                        delta=round(d,4), ci95=round(1.96*se,4), p=round(p,3),
                        sig='*' if p < 0.05 else ''))
    t = pd.DataFrame(out).sort_values('delta', ascending=False)
    print(t.to_string(index=False))
    n_sig = int((t.p < 0.05).sum())
    print(f'\nlabels where micro moves AP at p<0.05: {n_sig}/13'
          f'   (Bonferroni over 13 labels wants p<{0.05/13:.4f})')
    pn = t[t.label == 'Pneumonia'].iloc[0]
    print(f'Pneumonia: {pn.labs:.4f} -> {pn.plus_micro:.4f}  delta {pn.delta:+.4f} '
          f'+/-{pn.ci95:.4f}  p={pn.p:.3f}')
    print('  -> the single-run +0.0156 survives only if this CI excludes 0 AND p holds up')
    print('     against the 13-label Bonferroni bar; otherwise it was multiple comparisons.')
else:
    print('RUN_SEEDS=False. Set True to get error bars.')
    print(f'cost: {len(SWEEP)} configs x {len(SEEDS)} seeds = {len(SWEEP)*len(SEEDS)} runs')

  none      seeds [np.float64(0.391), np.float64(0.391), np.float64(0.3927)]  mean 0.3916
  concat    seeds [np.float64(0.4029), np.float64(0.4046), np.float64(0.4051)]  mean 0.4042
  film_db   seeds [np.float64(0.4037), np.float64(0.4079), np.float64(0.406)]  mean 0.4059
  film_bce  seeds [np.float64(0.4151), np.float64(0.4156), np.float64(0.415)]  mean 0.4152
  film_asl  seeds [np.float64(0.4192), np.float64(0.4194), np.float64(0.4221)]  mean 0.4202
  D3_labs   seeds [np.float64(0.4047), np.float64(0.4057), np.float64(0.403)]  mean 0.4045
  D3_12     seeds [np.float64(0.4039), np.float64(0.4031), np.float64(0.406)]  mean 0.4043

===== macro mAP over seeds (mean +/- 95% CI) =====
  config   mean   ci95     lo     hi  n
    none 0.3916 0.0025 0.3891 0.3941  3
  concat 0.4042 0.0028 0.4014 0.4070  3
 film_db 0.4059 0.0051 0.4007 0.4110  3
film_bce 0.4152 0.0008 0.4144 0.4161  3
film_asl 0.4202 0.0041 0.4162 0.4243  3
 D3_labs 0.4045 0.0034 0.4011 0.4079  3
   D3_12 0.4043 0.0037 0.4006 